# Chunking clinical guidelines: worked solution

Worked solution for the exercise in the lesson **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)** (*Retrieval-augmented generation (RAG)* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](02-chunking-clinical-guidelines.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Chunking splits documents into the passages a RAG system retrieves. If chunks are cut at arbitrary word counts, a clinical rule can be split in half, so the retrieved passage says when not to start a medicine but not when to stop it. Chunking that follows the document's structure, keeps each chunk's heading with it, and adds overlap where needed produces passages that stand alone and can be cited.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### The teaching document

A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

### Fixed-size chunking

Continuing from the protocol block: 40-word chunks.

In [ ]:
def fixed_size_chunks(text, size=40):
    words = text.split()
    return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

chunks = fixed_size_chunks(PROTOCOL)
print(f"{len(chunks)} chunks of up to 40 words\n")
for i, c in enumerate(chunks[2:5], start=2):
    print(f"[{i}] {c}\n")

# 5 chunks of up to 40 words
#
# [2] eGFR falls below 30 during treatment, stop metformin. ## 3. Foot care Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness,
#
# [3] a suspected infection of the bone, or a cold, pale foot. ## 4. Low blood glucose A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still
#
# [4] below 70 mg/dL. Review any medicine that may have caused the low glucose. ## 5. Eye screening Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.

### Section-aware chunking

Continuing from the protocol block: one chunk per '##' section, with title and heading kept.

In [ ]:
import re

def section_chunks(text):
    """Split on '## ' headings; keep the heading with its text so each chunk stands alone."""
    title = text.splitlines()[0].lstrip("# ").strip()
    parts = re.split(r"\n(?=## )", text)[1:]
    return [{"id": f"s{i + 1}", "text": f"{title} > {p.strip()}"} for i, p in enumerate(parts)]

sections = section_chunks(PROTOCOL)
print(f"{len(sections)} section chunks")
print(sections[1]["text"])

# 5 section chunks
# Synthetic diabetes clinic protocol (teaching document, not clinical guidance) > ## 2. Kidney function and metformin
# Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

### The difference at retrieval time

Continuing from the two previous blocks: retrieve the best chunk under each method.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def top_chunk(chunk_texts, question):
    vec = TfidfVectorizer(stop_words="english").fit(chunk_texts + [question])
    scores = cosine_similarity(vec.transform([question]), vec.transform(chunk_texts))[0]
    return chunk_texts[scores.argmax()]

question = "When should metformin be stopped because of kidney function?"
print("Fixed-size chunk retrieved:\n ", top_chunk(fixed_size_chunks(PROTOCOL), question))
print("\nSection chunk retrieved:\n ", top_chunk([s["text"] for s in section_chunks(PROTOCOL)], question))

# Fixed-size chunk retrieved:
#   target for many non-pregnant adults is below 7%, adjusted for the individual. ## 2. Kidney function and metformin Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If
#
# Section chunk retrieved:
#   Synthetic diabetes clinic protocol (teaching document, not clinical guidance) > ## 2. Kidney function and metformin
# Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## Exercise

**Goal:** Compare chunking strategies on the synthetic protocol and measure which keeps clinical rules intact.

**Data:** Synthetic protocol in the notebook; optional openly licensed guideline excerpt (for example WHO publications under CC BY-NC-SA 3.0 IGO) (Created for this course; optional source under its own licence)

### Step 1

Reproduce fixed-size and section-aware chunking.

In [ ]:
fixed = fixed_size_chunks(PROTOCOL)
sections_text = [s["text"] for s in section_chunks(PROTOCOL)]
print(f"{len(fixed)} fixed-size chunks, {len(sections_text)} section chunks")

### Step 2

Add 10-word overlap to fixed-size chunking and check whether the metformin question now retrieves the full rule.

In [ ]:
def overlapping_chunks(text, size=40, overlap=10):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

FULL_RULE = ["should not be started if eGFR is below 45", "If eGFR falls below 30 during treatment, stop metformin"]
question = "When should metformin be stopped because of kidney function?"
for name, chunker in [("fixed", fixed_size_chunks), ("fixed + 10-word overlap", overlapping_chunks)]:
    chunk = top_chunk(chunker(PROTOCOL), question)
    print(f"{name:<24} full rule retrieved: {all(part in chunk for part in FULL_RULE)}\n   {chunk}\n")

The overlap repeats the last 10 words of each chunk at the start of the next, so a rule that straddles a
boundary is more likely to appear whole in some chunk. Here it still does not help: the chunk that wins the
retrieval ends at "...at least once a year. Metformin", so neither version returns the stop rule. Overlap
reduces boundary problems on average; it does not guarantee that the retrieved chunk is complete.

### Step 3

Write five questions whose answers depend on a condition ('unless', 'if', 'below') and check which method keeps each rule whole.

In [ ]:
conditional = [   # (question, the words that must stay together for the rule to be correct)
    ("Can metformin be started if eGFR is 40?", "Metformin should not be started if eGFR is below 45."),
    ("When does HbA1c move to every 6 months?", "Once targets are met and treatment is stable, check HbA1c every 6 months."),
    ("When should a foot problem be referred urgently?", "Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot."),
    ("When should the hypo treatment be repeated?", "Repeat if glucose is still below 70 mg/dL."),
    ("What HbA1c target applies to most adults?", "A common target for many non-pregnant adults is below 7%, adjusted for the individual."),
]
methods = {"fixed": fixed_size_chunks(PROTOCOL), "overlap": overlapping_chunks(PROTOCOL), "section": sections_text}
print(f"{'question':<50}" + "".join(f"{m:>9}" for m in methods))
for q, rule in conditional:
    print(f"{q:<50}" + "".join(f"{('whole' if rule in top_chunk(c, q) else 'missing'):>9}" for c in methods.values()))

"missing" means the retrieved chunk does not contain the whole rule. Fixed-size chunks lose the urgent-referral
rule (it crosses a 40-word boundary) and, without overlap, the HbA1c target. The condition ("if", "below") is
often the part that gets cut off, which turns a correct rule into a dangerous one.

Section chunks keep every rule whole because the rules never cross a heading. The one "missing" result for
section chunks is different: keyword retrieval picked the wrong section, because nothing in the protocol says
"hypo" and "repeated" does not match "Repeat". That is a retrieval problem, the subject of the next lesson.

### Step 4

Add a small table (for example glucose thresholds) to the protocol and design a chunking rule that keeps headers with each row.

In [ ]:
TABLE = """## 6. Glucose thresholds (synthetic teaching values)
| Measurement | Low | Target | High |
|---|---|---|---|
| Fasting glucose (mg/dL) | below 70 | 80-130 | above 180 |
| Glucose 2 h after a meal (mg/dL) | below 70 | below 180 | above 250 |
| HbA1c (%) | not applicable | below 7 | above 9 |
"""

def table_row_chunks(table_text):
    """One chunk per row, each carrying the section title and the column headers."""
    lines = [l for l in table_text.strip().splitlines() if l.strip()]
    title, header = lines[0].lstrip("# "), [h.strip() for h in lines[1].strip("|").split("|")]
    chunks = []
    for row in lines[3:]:
        cells = [c.strip() for c in row.strip("|").split("|")]
        chunks.append(f"{title}: {cells[0]}: " + "; ".join(f"{h} {v}" for h, v in zip(header[1:], cells[1:])))
    return chunks

row_chunks = table_row_chunks(TABLE)
for c in row_chunks:
    print(c)
q = "What fasting glucose counts as high?"
print(f"\nQ: {q}\nA chunk: {top_chunk(sections_text + row_chunks, q)}")

A table split into fixed-size word chunks loses its headers, leaving "below 70 | 80-130 | above 180" with no
way to tell which number is which. Turning each row into a sentence that names its columns keeps every chunk
self-explanatory.

### Step 5

Optional: chunk a short openly licensed guideline excerpt and review the extracted text for errors.

Optional. Take a short excerpt from an openly licensed guideline (check the licence: many WHO publications are
CC BY-NC-SA 3.0 IGO, which allows non-commercial reuse with attribution). If it is a PDF, extract the text
(for example with `pypdf`), then chunk it by headings as above and *read the extracted text*. Typical problems:
headers and footers repeated inside paragraphs, words split across lines with hyphens, tables flattened into
a stream of numbers, footnote markers attached to doses ("500 mg1"), and two-column layouts read across
columns. Each one can change the meaning of a rule, so extraction errors must be fixed before indexing.

---

Lesson: [https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.